# Distant Contradiction Auditor - Local Legal-BERT Training with Early Stopping

**Runtime**: Local CPU or CUDA GPU

This notebook fine-tunes `nlpaueb/legal-bert-base-uncased` on the pairs dataset in the repository's `artifacts/` directory. It includes strict contract-stratified **Train / Val (Dev) / Test** evaluation and **Early Stopping** with a patience of 3 epochs.

Run the cells in order with the repository Python environment selected as the notebook kernel. Install dependencies once with `pip install -r requirements.txt` if needed.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

search_bases = [Path.cwd(), *Path.cwd().parents, Path.home() / 'OneDrive' / 'Desktop' / 'legal_project']
project_candidates = [
    candidate
    for base in search_bases
    for candidate in (base, base / 'contract-graph-risk')
]
PROJECT_ROOT = next(
    (candidate for candidate in project_candidates if (candidate / 'artifacts').is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find the repository root. Open this notebook from the contract-graph-risk workspace.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch

required_modules = ('datasets', 'transformers', 'accelerate', 'sklearn')
missing_modules = [
    module for module in required_modules if importlib.util.find_spec(module) is None
]
if missing_modules:
    print('Installing missing packages:', ', '.join(missing_modules))
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-r',
        str(PROJECT_ROOT / 'requirements.txt'),
    ])

try:
    import pandas  # noqa: F401
    import sklearn  # noqa: F401
except (ImportError, ValueError) as exc:
    print('Repairing incompatible NumPy/scientific Python packages:', exc)
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        'numpy<2',
        'pandas>=2.2,<3',
        'scikit-learn>=1.4',
    ])
    raise RuntimeError(
        'Dependencies were repaired. Restart the notebook kernel, then run the cells again.'
    ) from exc

ARTIFACTS = PROJECT_ROOT / 'artifacts'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Project root:', PROJECT_ROOT)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Project root: C:\Users\madhu\OneDrive\Desktop\legal_project\contract-graph-risk
Device: cpu


In [2]:
from pathlib import Path
import gzip
import shutil


def _unzip(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(src, 'rb') as inf, dst.open('wb') as out:
        shutil.copyfileobj(inf, out)


plain_candidates = [
    ARTIFACTS / 'pairs_colab_full.jsonl',
    ARTIFACTS / 'pairs_colab.jsonl',
    ARTIFACTS / 'pairs.jsonl',
]
gz_candidates = [
    ARTIFACTS / 'pairs_colab_full.jsonl.gz',
    ARTIFACTS / 'pairs_colab.jsonl.gz',
]

PAIRS = next((path for path in plain_candidates if path.exists()), None)
if PAIRS is None:
    gz_path = next((path for path in gz_candidates if path.exists()), None)
    if gz_path is not None:
        PAIRS = gz_path.with_suffix('')
        _unzip(gz_path, PAIRS)

if PAIRS is None or not PAIRS.exists():
    expected = ', '.join(path.name for path in plain_candidates + gz_candidates)
    raise FileNotFoundError(
        f'No pairs dataset found in {ARTIFACTS}. Expected one of: {expected}'
    )

OUT = ARTIFACTS / 'legalbert-dca'
print('Loaded dataset:', PAIRS, 'bytes:', PAIRS.stat().st_size)
print('Model output directory:', OUT)

Loaded dataset: C:\Users\madhu\OneDrive\Desktop\legal_project\contract-graph-risk\artifacts\pairs_colab_full.jsonl bytes: 286884126
Model output directory: C:\Users\madhu\OneDrive\Desktop\legal_project\contract-graph-risk\artifacts\legalbert-dca


In [3]:
import json
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support

MODEL = 'nlpaueb/legal-bert-base-uncased'
MAX_LEN = 256

rows = [json.loads(l) for l in PAIRS.read_text(encoding='utf-8', errors='replace').splitlines() if l.strip()]
by = {}
for r in rows:
    split_name = r.get('split', 'train')
    by.setdefault(split_name, []).append({
        'text_a': r['text_a'][:1200],
        'text_b': r['text_b'][:1200],
        'label': int(r['label']),
        'gap_bin': r.get('gap_bin', 'unknown'),
        'type': r.get('type', 'Inconsistency'),
    })
ds = DatasetDict({k: Dataset.from_list(v) for k, v in by.items()})
print('Dataset splits:', ds)

tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LEN, padding=False)

enc = ds.map(tokenize, batched=True, remove_columns=[c for c in ds['train'].column_names if c not in ('label', 'gap_bin', 'type')])
enc = enc.rename_column('label', 'labels')

Dataset splits: DatasetDict({
    train: Dataset({
        features: ['text_a', 'text_b', 'label', 'gap_bin', 'type'],
        num_rows: 114424
    })
    dev: Dataset({
        features: ['text_a', 'text_b', 'label', 'gap_bin', 'type'],
        num_rows: 15552
    })
    test: Dataset({
        features: ['text_a', 'text_b', 'label', 'gap_bin', 'type'],
        num_rows: 13344
    })
})


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

C:\Users\madhu\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\madhu\.cache\huggingface\hub\models--nlpaueb--legal-bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/114424 [00:00<?, ? examples/s]

Map:   0%|          | 0/15552 [00:00<?, ? examples/s]

Map:   0%|          | 0/13344 [00:00<?, ? examples/s]

In [5]:
import inspect

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {'precision': float(p), 'recall': float(r), 'f1': float(f1)}


use_cuda = DEVICE == 'cuda'
train_batch_size = 8 if use_cuda else 2
eval_batch_size = 16 if use_cuda else 4
training_parameters = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs = {
    'output_dir': str(OUT / 'checkpoints'),
    'per_device_train_batch_size': train_batch_size,
    'per_device_eval_batch_size': eval_batch_size,
    'gradient_accumulation_steps': 4,
    'num_train_epochs': 6,
    'learning_rate': 2e-5,
    'lr_scheduler_type': 'cosine',
    'weight_decay': 0.01,
    'fp16': use_cuda,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'greater_is_better': True,
    'save_total_limit': 2,
    'logging_steps': 50,
    'dataloader_num_workers': 0,
    'report_to': [],
    'seed': 13,
}

if 'warmup_ratio' in training_parameters:
    training_kwargs['warmup_ratio'] = 0.1
else:
    steps_per_epoch = max(1, len(enc['train']) // (train_batch_size * 4))
    training_kwargs['warmup_steps'] = max(1, int(steps_per_epoch * 6 * 0.1))

if 'eval_strategy' in training_parameters:
    training_kwargs['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in training_parameters:
    training_kwargs['evaluation_strategy'] = 'epoch'

args = TrainingArguments(**training_kwargs)

eval_ds = enc['dev'] if 'dev' in enc else enc['test']
callbacks = [EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
trainer_parameters = inspect.signature(Trainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': enc['train'],
    'eval_dataset': eval_ds,
    'compute_metrics': compute_metrics,
    'callbacks': callbacks,
}
if 'processing_class' in trainer_parameters:
    trainer_kwargs['processing_class'] = tok
else:
    trainer_kwargs['tokenizer'] = tok

trainer = Trainer(**trainer_kwargs)

print(f'Starting fine-tuning on {DEVICE} (patience=3)...')
trainer.train()
OUT.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUT)
tok.save_pretrained(OUT)
print('Saved fine-tuned best checkpoint to:', OUT)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Starting fine-tuning on cpu (patience=3)...


C:\Users\madhu\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print('=== Final Evaluation on Held-Out Test Set ===')
test_eval = trainer.predict(enc['test'])
logits, labels = test_eval.predictions, test_eval.label_ids
preds = np.argmax(logits, axis=-1)
p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)

test_rows = by['test']
by_bin = {}
for i, row in enumerate(test_rows):
    if row['label'] != 1:
        continue
    b = row['gap_bin']
    by_bin.setdefault(b, {'yt': [], 'yp': []})
    by_bin[b]['yt'].append(1)
    by_bin[b]['yp'].append(int(preds[i]))

gap = {k: float(f1_score(v['yt'], v['yp'], zero_division=0)) for k, v in by_bin.items()}
metrics = {
    'model': MODEL,
    'device': DEVICE,
    'test_precision': float(p),
    'test_recall': float(r),
    'test_f1': float(f1),
    'f1_by_gap_bin_gold_positives': gap,
    'n_test': len(test_rows),
}

metrics_path = ARTIFACTS / 'metrics_legalbert.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))
print('Wrote evaluation metrics to:', metrics_path)